# Visualize Outputs of CE Inference

In [ ]:
import itertools
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchvision import models, transforms
from tqdm.notebook import tqdm

# Set the project directory
CURR_PROJECT_DIR: str = '/workspace/current/loce'
sys.path.append(CURR_PROJECT_DIR)

from bg_randomized_loce.utils.consts import *
from bg_randomized_loce.utils import consts
PROJECT_DIR = consts.PROJECT_DIR = CURR_PROJECT_DIR

from bg_randomized_loce.utils.loce_storage_helpers import all_pkls_to_npz
from bg_randomized_loce.data_structures.mscoco import SegmentationDataset
from bg_randomized_loce.utils.eval_util import with_globalized_ces, df_where_cols_equal, df_from_csv, to_layer_depth_vals, plot_means
from bg_randomized_loce.loce.loce_utils import get_projection, get_rgb_binary_mask
from bg_randomized_loce.background_pasting.background_pasting import BGType, PasteOnBackground

%load_ext autoreload
%autoreload 2

In [ ]:
## SETTINGS
fg_dataset_key = 'pascal_voc'
concept_ids = [
    2, # bicycle (vehicles)
    6, # bus
    17, # sheep (animals)
    18, # sofa (furniture)
]
variants = {"ce_method": ['net2vec_proper_bce', 'globalized_loce_proper_bce'],
            "bg_randomizer_key": ['vanilla', 'places', ]}#'places_voronoi', 'synthetic']}
vis_bg_types = [BGType.voronoi]  # The background types to visualize
num_vis_per_concept = 1
top_quantile = 0.5
model_key = 'efficientnet'

# Settings dict
common_settings = {
    DATA: fg_dataset_key,
    NUM_BG: 1,
    MODEL: model_key,
    DEPTH: 'late',
}

# Paths to the caches with IoU results
csv_caches = [os.path.join(PROJECT_DIR, reldir) for reldir in [
    'results/results/iou/cache',
]]

# Paths to the LoCEs (and their npz caches)
pkl_npz_dirs: list[tuple[str, str]] = [
    (os.path.join(PROJECT_DIR, pkl_dir), os.path.join(PROJECT_DIR, npz_dir))
    for pkl_dir, npz_dir in [
        ("results/results/pkl", "results/results/npz"),
]]

## Load IoUs and select model_id, layer, img_id

In [ ]:
csv_files = [f for csv_cache in csv_caches for f in list(Path(csv_cache).glob('**/results_*.csv'))]
assert len(csv_files) > 0
ious_list = []
it = tqdm(csv_files)
for csv_path in it:
    it.set_description(str(csv_path))
    curr_ious = df_from_csv(csv_path)
    curr_ious = curr_ious[curr_ious[CE_METHOD].isin(variants[CE_METHOD]) &  curr_ious[BG].isin(variants[BG])]
    ious_list.append(curr_ious)
print("Concatenating ...")
ious = pd.concat(ious_list, axis=0, ignore_index=True)

# add depth column
print("Adding layer depth ...")
ious[DEPTH] = to_layer_depth_vals(ious)

print("Filtering out unneeded variants ...")
ious = ious[(ious[CE_METHOD] != LOCE_OLD) & (ious[CE_METHOD] != NET2VEC_OLD)]

ious

In [ ]:
restr = {k: v for k, v in {**common_settings}.items() if k not in [MODEL, DATA]}
fig, means, std = plot_means(ious, restrict_to=restr,
    values=IOU,
    compare=[BG], group_cols=[CE_METHOD],
    side_by_side=[MODEL],
    top_to_bottom=[DATA, DEPTH],
    legend=False,
    pretty_names={**PRETTY_NAMES, 'pascal_voc': "VOC", 'imagenets50': "ImageNetS50", "places": "Places", "any": "tested on randomized bg", "vanilla_tested": "tested on vanilla bg"},
    axsize=(3,2),
    rot=40
)
fig.suptitle(f"{restr}")
fig.axes[0].legend(loc='right', bbox_to_anchor=(1.1, 1.2)) #outside upper right


### Select model_key and depth

In [ ]:
# Get the model & layer for which vanilla works best on average on vanilla
restrictions = {BG: "vanilla", TEST_BG: "vanilla", NUM_BG: 1, DATA: "pascal_voc"}
print("Maximum for", restrictions)

tmp = (df_where_cols_equal(ious, restrictions)
 .groupby([MODEL, DEPTH])
 .iou.mean()
)
tmp = tmp[tmp == tmp.max()].reset_index()
display(tmp)
model_key, depth = tmp.model_key.unique()[0], tmp.depth.unique()[0]
common_settings |= {MODEL: model_key, DEPTH: depth}

### Select img_id

In [ ]:
# Get the img_id for which vanilla works best on average on vanilla
def get_img_ids_for(ious: pd.DataFrame, restrictions: dict,
                    k: int = None, top_k: int = None, top_quantile: float = None, 
                    verbose=False) -> pd.Series:
        mean_iou_per_img = (df_where_cols_equal(ious, restrictions)
                .groupby(TEST_IMG_ID)
                .iou.mean()
        )
        if verbose:
             mean_iou_per_img.plot.box(); plt.show()
        if top_quantile is not None:
             mean_iou_per_img = mean_iou_per_img[mean_iou_per_img>mean_iou_per_img.quantile(top_quantile)]
        if top_k is not None:
             mean_iou_per_img = mean_iou_per_img.loc[:min(len(mean_iou_per_img.index), top_k)]
        if k is not None and len(mean_iou_per_img.index)>=k:
             idx = random.sample(mean_iou_per_img.index.to_list(), k=k)
             mean_iou_per_img = mean_iou_per_img.loc[idx]

        if verbose:
             display(mean_iou_per_img
                        .sort_values()
                        .reset_index()
                        # show value decay
                        .style.background_gradient(subset=[IOU])
                        # highlight what is closeby to the chosen image
                        #.highlight_between(subset=[IOU], left=tmp[img_id]-0.01, right=tmp[img_id]+0.01)
                        )        
        return mean_iou_per_img


# Show some exemplary image selection
restrictions |= {MODEL: model_key, DEPTH: depth}
print("Boxplot for", restrictions)
selected_ious: pd.Series = get_img_ids_for(ious, restrictions, top_quantile=0.5, verbose=True)
img_ids = selected_ious.index.to_list()


## Load concept embeddings

In [ ]:
# Paths
ces_list = []
for pkl_dir, npz_dir in pkl_npz_dirs:
    # load all concept embeddings + meta-info
    curr_ces = pd.DataFrame(await all_pkls_to_npz(pkl_dir, npz_dir=npz_dir, verbose=True))
    curr_ces: pd.DataFrame = with_globalized_ces(curr_ces.copy(), experiment_setting_cols=EXPERIMENT_SETTING_COLS, add_depth=True)
    ces_list.append(curr_ces)
ces = pd.concat(ces_list, axis=0, ignore_index=True)

# some validation
display(ces.columns, len(ces.index))
for col in EXPERIMENT_SETTING_COLS: print(col, ces[col].unique())


In [ ]:
selected_ces = []
for ce_method, bg_randomizer_key in itertools.product(*variants.values()):
    #settings = {CE_METHOD: ce_method, BG: bg_randomizer_key, **common_settings}
    selected_ces.append(df_where_cols_equal(ces, common_settings))
selected_ces: pd.DataFrame = pd.concat(selected_ces)
selected_ces

## Set up Model

In [ ]:
from bg_randomized_loce.loce.loce_utils import LoCEActivationsTensorExtractor

layer = selected_ces.layer.unique()[0] # should be just one
model_builder = WRAPPED_MODELS[common_settings[MODEL]]
propagator, image_processor = model_builder([layer], device='cpu')
activations_extractor = LoCEActivationsTensorExtractor(propagator, model_key, processor=image_processor)

## Single image case: Set up Dataset and Test Image

In [ ]:
# # Set up dataset
# fg_data: SegmentationDataset = TEST_DATA_BUILDERS[fg_dataset_key](category_ids=[concept_id], device='cpu')
# print("Available datasets:", [d for d in TEST_DATA_BUILDERS.keys()])
# print(f"Available concepts for {fg_dataset_key}:", [(c_id, c_name) for c_id, c_name in fg_data.ALL_CAT_NAMES_BY_ID.items()
#                                                     if c_id != 'ignore'])
# print(f"Available image IDs:", fg_data.img_ids)

### Single image case: Sample, load and visualize image

In [ ]:
# img_pil, gt_mask = fg_data[img_id]

# print(f"Chosen sample: {fg_dataset_key=}, {concept_id=}, {concept_name=}, {len(fg_data)=}, {img_id=}")
# for iid in [img_id]:#fg_data.img_ids[:10]:
#     img_pil, gt_mask = fg_data[iid]
#     print(f"{gt_mask.size()=}, {img_pil.size=}")
#     display(img_pil)
#     display(transforms.ToPILImage()(gt_mask.to(torch.float)))


## Multi-image case: Get Images

In [ ]:
# For fixed concept_id:
#imgs_pil, masks, concepts = zip(*[(*TEST_DATA_BUILDERS[fg_dataset_key](category_ids=[concept_id], device='cpu')[i], concept_id) for concept_id in concept_ids])
#imgs_pil, masks, concepts = zip(*[(*fg_data[i], concept_id) for i in img_ids])
#concept_ids = [2, 6] # bicycle, bus

random.seed(1)

bg_data = BG_DATA_BUILDERS["places"](device='cpu')
img_ids, masks, concepts, imgs_pil_by_bg  = [], [], [], {BGType.original: []} | {bg_type: [] for bg_type in vis_bg_types}
for concept_id in concept_ids:
    fg_data: SegmentationDataset = TEST_DATA_BUILDERS[fg_dataset_key](category_ids=[concept_id], device='cpu')
    # To randomly sample images from the top 50% performing test image samples uncomment the following instead:
    # curr_img_ids = get_img_ids_for(ious, {**common_settings, BG: "vanilla", TEST_BG: "vanilla", CAT: concept_id},
    #                           top_quantile=top_quantile, k=num_vis_per_concept,
    #                           ).index.to_list()
    curr_img_ids = random.sample(fg_data.img_ids, k=num_vis_per_concept)
    img_ids.extend(curr_img_ids)

    # Load images and masks
    for img_id in curr_img_ids:
        img_pil, mask = fg_data[img_id]
        concepts.append(fg_data.ALL_CAT_NAMES_BY_ID[concept_id])
        imgs_pil_by_bg[BGType.original].append(img_pil)
        masks.append(mask)

    # Load bg-randomized images
    forbidden_classes = [imagenet_c for cname in fg_data.cat_name_by_id.values()
                         for imagenet_c in AS_IMAGENET_IDS_OR_NAMES.get(cname, [])] or None
    for bg_type in vis_bg_types:
        bg_paster: PasteOnBackground = PasteOnBackground(
            background_loader=bg_data, bg_type=bg_type,
            num_imgs=1,
            forbidden_classes_bg=forbidden_classes,)

        # set the transform
        fg_data.transform = bg_paster
        for img_id in curr_img_ids:
            imgs_pil_by_bg[bg_type].append(fg_data[img_id][0][0])


## Get Activations

In [ ]:
# Get activations
acts_by_bg = {bg_type: list(map(lambda img_pil: activations_extractor.get_bchw_acts_preds_dict(
                                image_pil=img_pil,
                                get_predictions=False
                            )[0][layer].squeeze(0),
                            imgs_pil))
        for bg_type, imgs_pil in imgs_pil_by_bg.items()}

layer

In [ ]:
# Check whether the shapes match (they should)
#{c: a[0].size() for c, a in acts_by_bg.items()}, selected_ces.ce.apply(lambda s: s.shape).unique()

In [ ]:
df_where_cols_equal(ces, {**common_settings, CE_METHOD: NET2VEC, BG: VANILLA, CAT: '1'})

In [ ]:
from torchvision.transforms import functional as F


def plot_variants(ce_methods: list[str], bg_randomizer_keys: list[str], vis_bg_types=vis_bg_types): # TODO: properly hand over arguments
    """Plot a grid of images depicting the CE predictions for specified variants of CE methods and backgrounds.
    """
    # For each image show: original, ground truth mask, and for each variant the mask with and without bg randomization
    grid_size = (2 + len(vis_bg_types) + (len(vis_bg_types)+1) * len(ce_methods) * len(bg_randomizer_keys), len(img_ids))
    scale = 0.7
    img_ratio = (4, 3)  # (width, height)
    fig, axes = plt.subplots(grid_size[1], grid_size[0], figsize=(img_ratio[0]*grid_size[0]*scale, img_ratio[1]*grid_size[1]*scale), sharex=True, sharey=True)

    # Plot images and ground truth masks
    for i, (img_pil_by_bg, mask, concept) in enumerate(zip([{bg: imgs_pil_by_bg[bg][i] for bg in imgs_pil_by_bg.keys()} for i in range(len(img_ids))], masks, concepts)):
        img_pil = img_pil_by_bg.pop(BGType.original)
        axes[i][0].imshow(img_pil)
        if i == 0: axes[i][0].set_title("original")
        axes[i][0].set_ylabel(f"{concept}")

        # For a different overlay variant use the following:
        # gt_rgb = get_rgb_binary_mask(mask.cpu().numpy(), target_size=img_pil.size)
        # overlay = (0.6 * np.array(img_pil) + 0.4 * gt_rgb).astype(np.uint8)
        gt_rgb = np.stack([np.array(F.to_pil_image(mask.to(torch.uint8)).resize(img_pil.size))]*3, axis=-1)
        overlay = (np.array(img_pil) * (0.8* gt_rgb + 0.2)).astype(np.uint8)

        axes[i][1].imshow(overlay)
        if i == 0: axes[i][1].set_title("ground truth")

        for j, (test_bg, img_pil_rand) in enumerate(img_pil_by_bg.items()):
            axes[i][j+2].imshow(img_pil_rand)
            if i == 0: axes[i][j+2].set_title(f"{test_bg.name} rand.")
        
        
    # Get and plot predicted masks 
    for i, (img_act_by_bg, cat_id) in enumerate(zip([{test_bg: (imgs_pil[i], acts_by_bg[test_bg][i])
                                                        for test_bg, imgs_pil in imgs_pil_by_bg.items()}
                                                    for i in range(len(img_ids))], concept_ids)):
        num_bgs = len(img_act_by_bg.keys())
        j_offset = 1 + num_bgs
        for j, (ce_meth, bg_key) in enumerate(itertools.product(ce_methods, bg_randomizer_keys)):
            #ce: np.ndarray = np.array(final_ces[(ce_meth, bg_key)][CE].item()).reshape(-1)
            ce: np.ndarray = df_where_cols_equal(ces, {CE_METHOD: ce_meth, BG: bg_key, CAT: str(cat_id), **common_settings}).sample(1)[CE].item()
            
            for h, (test_bg, (img_pil, act)) in enumerate(img_act_by_bg.items()):
                ax = axes[i][j_offset + j*num_bgs + h]
                if i == 0: ax.set_title(f"{'Net2Vec' if ce_meth.split("_")[0]=='net2vec' else 'GloCE'} "
                                        f"{'rand., ' if bg_key != 'vanilla' else ''}on {test_bg.name}")
                
                pred_uint8: np.ndarray = get_projection(ce, act.cpu().numpy())
                #display(Image.fromarray(pred_uint8).resize(img_pil.size))
                pred_rgb = get_rgb_binary_mask(pred_uint8, target_size=img_pil.size)
                overlay = (0.6 * np.array(img_pil) + 0.4 * pred_rgb).astype(np.uint8)
                
                ax.imshow(Image.fromarray(overlay))

    # Style stuff
    for i in range(len(axes)):
        for j in range(len(axes[i])):
            ax = axes[i][j]
            ax.set_xticks([])
            ax.set_yticks([])
    fig.set_tight_layout(tight=True)
    fig.tight_layout()
    fig.show()

    return fig



fig1 = plot_variants(ce_methods=['net2vec_proper_bce',],# 'globalized_loce_proper_bce'],
                     bg_randomizer_keys=['vanilla', 'places', ]#'places_voronoi', 'synthetic']
)
plt.savefig(os.path.join(PROJECT_DIR, "results/resources/examples-net2vec.pdf"), bbox_inches='tight')
fig2 = plot_variants(ce_methods=["globalized_loce_proper_bce"],
                     bg_randomizer_keys=['vanilla', 'places'])
plt.savefig(os.path.join(PROJECT_DIR, "results/resources/examples-gloce.pdf"), bbox_inches='tight')